In [13]:
# 连接数据库

import sqlite3

db_path = "db/bible.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("数据库连接成功")

数据库连接成功


In [8]:
# 快速备份

import sqlite3

src = "db/bible.db"
backup = "db/bible_backup.sqlite"

with sqlite3.connect(src) as src_conn:
    with sqlite3.connect(backup) as bak_conn:
        src_conn.backup(bak_conn)

print("SQLite 备份完成")

SQLite 备份完成


In [5]:
# 快速恢复

import sqlite3

backup = "db/bible_backup.sqlite"
target = "db/bible.db"

with sqlite3.connect(backup) as src_conn:
    with sqlite3.connect(target) as dst_conn:
        src_conn.backup(dst_conn)

print("数据库恢复完成")

数据库恢复完成


In [30]:
# 填充指定章（手动）的 tokens 表，从 verse 表获取数据

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(book_abbr: str, book_id: int, chapter: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"📖 处理章节：{book_abbr} {chapter}")

    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"

            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None
            ))

    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # ✅ 填充 word_seq 和 align_ID（只改这里）
    print("🔢 正在填充 word_seq / align_id...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY chapter, verse, token_id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_id:02d}_{book_abbr}_{chapter:03d}__{seq:04d}"

        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()

    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    build_tokens_for_chapter(
        book_abbr="Gen",
        book_id=1,
        chapter=4
    )

📖 处理章节：Gen 4
  需要处理的 verse 数：26
✅ 插入 token 数：1293
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：597
🎉 本章 token 构建完成


In [26]:
# 填充指定章（手动）的 tokens 表，从 verse 表获取数据
# 修复brother's被分词的情况，brother's不要分词，因为强制对齐的时候它是看作一个单词的

import sqlite3
import re
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    # 1. 归一化智能引号
    text = (
        text.replace("‘", "'")
            .replace("’", "'")
            .replace("ʼ", "'")
    )

    # 2. 合并所有格
    pattern = re.compile(r"\b([A-Za-z]+)('[sS])\b")
    text = pattern.sub(lambda m: m.group(1) + "\u200B" + m.group(2), text)

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if not current:
            return

        if "\u200B" in current:
            # ✅ 只还原所有格，不改其它字符
            current = current.replace("\u200B", "").replace("'S", "'s")
            current_type = "word"

        tokens.append((current, current_type))
        current = ""
        current_type = None

    for ch in text:
        if ch == "\u200B":
            continue

        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(book_abbr: str, book_id: int, chapter: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"📖 处理章节：{book_abbr} {chapter}")

    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"

            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None
            ))

    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # ✅ 填充 word_seq 和 align_id
    print("🔢 正在填充 word_seq / align_id...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY chapter, verse, token_id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_id:02d}_{book_abbr}_{chapter:03d}__{seq:04d}"

        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()

    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    build_tokens_for_chapter(
        book_abbr="Gen",
        book_id=1,
        chapter=4
    )

📖 处理章节：Gen 4
  需要处理的 verse 数：26
❌ 拼接不一致
verse: Gen.4.9
原句: 'Then the Lord said to Cain, “Where is your brother Abel? ” He said, “I do not know; am I my brother’s keeper? ”'
拼接: "Then the Lord said to Cain, “Where is your brother Abel? ” He said, “I do not know; am I my brother's keeper? ”"
❌ 拼接不一致
verse: Gen.4.10
原句: 'And the Lord said, “What have you done? Listen; your brother’s blood is crying out to me from the ground!'
拼接: "And the Lord said, “What have you done? Listen; your brother's blood is crying out to me from the ground!"
❌ 拼接不一致
verse: Gen.4.11
原句: 'And now you are cursed from the ground, which has opened its mouth to receive your brother’s blood from your hand.'
拼接: "And now you are cursed from the ground, which has opened its mouth to receive your brother's blood from your hand."
❌ 拼接不一致
verse: Gen.4.21
原句: 'His brother’s name was Jubal; he was the ancestor of all those who play the lyre and pipe.'
拼接: "His brother's name was Jubal; he was the ancestor of all those who pl

In [28]:
import sqlite3
import re
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    """
    英文分词（最终稳定版）：
    - 保留原文引号（‘ ’ “ ” '）
    - brother's / brother’s 视为一个 word
    - 绝不拆成 brother ' s
    """
    if not text:
        return []

    # ✅ 不归一化引号，原样保留
    pattern = re.compile(r"\b([A-Za-z]+)(’[sS]|'[sS])\b")
    text = pattern.sub(lambda m: m.group(1) + "\u200B" + m.group(2), text)

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if not current:
            return

        if "\u200B" in current:
            current = current.replace("\u200B", "")
            current_type = "word"

        tokens.append((current, current_type))
        current = ""
        current_type = None

    for ch in text:
        if ch == "\u200B":
            continue

        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(book_abbr: str, book_id: int, chapter: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"📖 处理章节：{book_abbr} {chapter}")

    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"

            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None,   # entity_key
                None,   # word_seq（后面 UPDATE）
                None    # align_id（后面 UPDATE）
            ))

    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key,
            word_seq,
            align_id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # ✅ 填充 word_seq 和 align_id
    print("🔢 正在填充 word_seq / align_id...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY chapter, verse, token_id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_id:02d}_{book_abbr}_{chapter:03d}__{seq:04d}"

        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()

    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    build_tokens_for_chapter(
        book_abbr="Gen",
        book_id=1,
        chapter=4
    )

📖 处理章节：Gen 4
  需要处理的 verse 数：26
✅ 插入 token 数：1293
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：597
🎉 本章 token 构建完成


In [ ]:
# mp3 -> wav 
# 增量更新，要改成指定章，为了工作流

import os
import subprocess
import time

MP3_DIR = "outputs/mp3"
WAV_DIR = "outputs/wav"

def batch_mp3_to_wav(mp3_dir: str, wav_dir: str, sample_rate: str = "16000"):
    if not os.path.isdir(mp3_dir):
        raise FileNotFoundError(f"❌ 输入目录不存在：{mp3_dir}")

    os.makedirs(wav_dir, exist_ok=True)

    # 以文件名（不含后缀）做差集
    mp3_files = {
        f[:-4] for f in os.listdir(mp3_dir)
        if f.lower().endswith(".mp3")
        and os.path.isfile(os.path.join(mp3_dir, f))
    }
    wav_files = {
        f[:-4] for f in os.listdir(wav_dir)
        if f.lower().endswith(".wav")
        and os.path.isfile(os.path.join(wav_dir, f))
    }

    to_convert = sorted(mp3_files - wav_files)
    skipped = len(mp3_files & wav_files)

    if skipped:
        print(f"⏭️ 已存在跳过：{skipped} 个")
    if not to_convert:
        print("✅ 全部已同步，无需转换")
        return

    print(f"🎬 待转换 {len(to_convert)} 个\n")

    total = len(to_convert)
    t_start = time.time()

    for i, name in enumerate(to_convert, 1):
        mp3_path = os.path.join(mp3_dir, f"{name}.mp3")
        wav_path = os.path.join(wav_dir, f"{name}.wav")

        cmd = [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", mp3_path,
            "-ar", sample_rate,
            "-ac", "1",
            wav_path
        ]

        s = time.time()
        subprocess.run(cmd, check=True)
        cost = time.time() - s

        print(f"[{i}/{total}] ✅ {name}.wav  ({cost:.2f}s)")

    print(f"\n🏁 完成！共转换 {total} 个，总耗时 {time.time() - t_start:.2f}s")


if __name__ == "__main__":
    batch_mp3_to_wav(MP3_DIR, WAV_DIR, sample_rate="16000")

In [ ]:
# 使用 plaintext 和 wav，输出对应章的 TextGrid
# 增量更新，要改成指定章，为了工作流

import os
import shutil
import subprocess

# ===== 目录 =====
WAV_DIR = "outputs/wav"
TXT_DIR = "outputs/plaintext"
CORPUS_DIR = "corpus"
ALIGN_OUT = "outputs/forcealign"

# ===== MFA 模型路径（✅ 关键修复）=====
dict_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/dictionary/english_us_arpa.dict"
)
acoustic_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/acoustic/english_us_arpa.zip"
)

# ===== 注入 aligner 环境 =====
conda_prefix = subprocess.check_output(
    ["conda", "info", "--base"], text=True
).strip()
aligner_bin = os.path.join(conda_prefix, "envs", "aligner", "bin")
os.environ["PATH"] = aligner_bin + ":" + os.environ["PATH"]

def prepare_and_align():
    os.makedirs(CORPUS_DIR, exist_ok=True)
    os.makedirs(ALIGN_OUT, exist_ok=True)

    # 已对齐结果
    aligned = {
        f[:-9] for f in os.listdir(ALIGN_OUT)
        if f.endswith(".TextGrid")
    }

    wavs = {f[:-4] for f in os.listdir(WAV_DIR) if f.endswith(".wav")}
    txts = {f[:-4] for f in os.listdir(TXT_DIR) if f.endswith(".txt")}

    ready = wavs & txts
    to_align = sorted(ready - aligned)

    if not to_align:
        print("✅ 所有样本已完成强制对齐")
        return

    print(f"🎯 待对齐：{len(to_align)} 个")

    for name in to_align:
        shutil.copy(
            os.path.join(WAV_DIR, f"{name}.wav"),
            os.path.join(CORPUS_DIR, f"{name}.wav")
        )
        shutil.copy(
            os.path.join(TXT_DIR, f"{name}.txt"),
            os.path.join(CORPUS_DIR, f"{name}.txt")
        )

    cmd = [
        "mfa", "align", CORPUS_DIR,
        dict_path,
        acoustic_path,
        ALIGN_OUT,
        "--clean", "--overwrite"
    ]

    print("🚀 开始强制对齐...\n")
    subprocess.run(cmd, check=True)
    print("\n🏁 强制对齐完成")

if __name__ == "__main__":
    prepare_and_align()

In [32]:
# 使用 TextGrid ，指定章 填充 timestamps 表
# 修复brother's问题

import re
import sqlite3
import os

DB_PATH = "db/bible.db"
TEXTGRID_DIR = "outputs/forcealign"

# ========= 交互输入 =========
book_abbr = input("请输入书卷简称（如 01_Gen）：").strip()
chapter = int(input("请输入章数（如 1）：").strip())

# =========================

filename = f"{book_abbr}_{chapter:03d}_en.TextGrid"
filepath = os.path.join(TEXTGRID_DIR, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(f"❌ 找不到文件: {filepath}")

with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()


# ======================
# 1️⃣ 只提取 item [1]
# ======================
item1_pattern = re.compile(
    r"item\s*\[1\]:(.*?)(?=\n\s*item\s*\[\d+\]:|\Z)",
    re.DOTALL
)

m = item1_pattern.search(content)
if not m:
    raise ValueError("❌ 未找到 item [1]，请检查 TextGrid 结构")

item1_content = m.group(1)


# ======================
# 2️⃣ 解析 intervals
# ======================
interval_pattern = re.compile(
    r"intervals\s*\[\d+\]:\s*\n"
    r"\s*xmin\s*=\s*([\d.]+)\s*\n"
    r"\s*xmax\s*=\s*([\d.]+)\s*\n"
    r'\s*text\s*=\s*"([^"]*)"',
    re.MULTILINE
)

matches = interval_pattern.findall(item1_content)


# ======================
# 3️⃣ 拆词逻辑（核心）
# ======================
def split_word(word: str):
    """
    brother's -> ["brother", "s"]
    其他 -> [word]
    """
    if word.endswith("'s"):
        return [word[:-2], "s"]
    return [word]


# ======================
# 4️⃣ 写入数据库
# ======================
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='timestamps'"
)
if cursor.fetchone() is None:
    raise RuntimeError("❌ timestamps 表不存在，请先建表")

word_seq = 0

for xmin, xmax, text in matches:
    text = text.strip()
    if text == "":
        continue

    words = split_word(text)
    duration = float(xmax) - float(xmin)

    # 时间均摊（你之前反对均分，这里先给基础版，下面我会给你“短 s”版本）
    per_word_duration = duration / len(words)

    for i, w in enumerate(words):
        word_seq += 1

        start = float(xmin) + i * per_word_duration
        end = start + per_word_duration

        start_ms = int(round(start * 1000, 0))
        end_ms = int(round(end * 1000, 0))

        align_id = f"{book_abbr}_{chapter:03d}__{word_seq:04d}"

        cursor.execute(
            """
            INSERT INTO timestamps (
                id, chapter, word_seq, word, start_time, end_time
            ) VALUES (?, ?, ?, ?, ?, ?)
            """,
            (align_id, chapter, word_seq, w, start_ms, end_ms)
        )

        print(f"{align_id:20s} | {w:10s} | {start_ms:>6} | {end_ms:>6}")

conn.commit()
conn.close()

print(f"\n✅ 共插入 {word_seq} 条记录（已处理 's 拆分）")

请输入书卷简称（如 01_Gen）：01_Gen
请输入章数（如 1）：4
01_Gen_004__0001     | now        |    140 |    340
01_Gen_004__0002     | the        |    340 |    450
01_Gen_004__0003     | man        |    450 |    820
01_Gen_004__0004     | knew       |    820 |    990
01_Gen_004__0005     | his        |    990 |   1210
01_Gen_004__0006     | wife       |   1210 |   1560
01_Gen_004__0007     | eve        |   1630 |   2160
01_Gen_004__0008     | and        |   2360 |   2520
01_Gen_004__0009     | she        |   2520 |   2660
01_Gen_004__0010     | conceived  |   2660 |   3330
01_Gen_004__0011     | and        |   3330 |   3490
01_Gen_004__0012     | bore       |   3490 |   3700
01_Gen_004__0013     | cain       |   3700 |   4240
01_Gen_004__0014     | saying     |   4360 |   4860
01_Gen_004__0015     | i          |   5240 |   5310
01_Gen_004__0016     | have       |   5310 |   5600
01_Gen_004__0017     | produced   |   5600 |   6060
01_Gen_004__0018     | a          |   6060 |   6110
01_Gen_004__0019     | man

In [11]:
import sqlite3

db_path = "db/bible.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

query = """
SELECT t.id, t.word, tk.token
FROM timestamps t
JOIN tokens tk ON t.id = tk.align_id
"""

cursor.execute(query)

for row in cursor.fetchall():
    id_name = row[0]
    word = (row[1] or "").strip().lower()
    token = (row[2] or "").strip().lower()

    if word != token:
        print(f"发现不一致，id 为: {id_name}")
        print(f"timestamps.word = {row[1]}")
        print(f"tokens.token    = {row[2]}")
        break
else:
    print("✅ 所有 id 对应的 word 和 token 完全一致")

conn.close()

发现不一致，id 为: 12_2K_004__0369
timestamps.word = own
tokens.token    = He


In [31]:
# 检查 align_id是否匹配（播放产生高亮漂移）

import sqlite3

db_path = "db/bible.db"

# ✅ 设置你要检查的 id 前缀
id_prefix = input("前缀").strip()

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

query = """
SELECT t.id, t.word, tk.token
FROM timestamps t
JOIN tokens tk ON t.id = tk.align_id
WHERE t.id LIKE ?
"""

cursor.execute(query, (f"{id_prefix}%",))

for row in cursor.fetchall():
    id_name = row[0]
    word = (row[1] or "").strip().lower()
    token = (row[2] or "").strip().lower()

    if word != token:
        print(f"发现不一致，id 为: {id_name}")
        print(f"timestamps.word = {row[1]}")
        print(f"tokens.token    = {row[2]}")
        break
else:
    print(f"✅ 所有以 '{id_prefix}' 开头的 id，其 word 与 token 完全一致")

conn.close()

前缀01_Gen_004
发现不一致，id 为: 01_Gen_004__0205
timestamps.word = brother's
tokens.token    = brother
